# Hippo Encoder Colab

Reset notebook for the original tiny-LLM idea:

- prepare a text-only dataset
- distill a tiny model against the frozen teacher encoder
- evaluate how well the tiny model matches the teacher geometry


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/CameronBadman/Hippo-encoder.git'
WORKDIR = Path('/content/Hippo-encoder')

%cd /content
if WORKDIR.exists():
    !rm -rf /content/Hippo-encoder

!git clone {REPO_URL} /content/Hippo-encoder
%cd /content/Hippo-encoder
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
short = subprocess.check_output(['git', 'log', '-1', '--oneline'], text=True).strip()
print('HEAD:', commit)
print('Latest:', short)


In [ ]:
%cd /content/Hippo-encoder
!python -m pip install --upgrade pip
!pip install -e .


In [ ]:
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/hippo_encoder_data')
RUN_ROOT = Path('/content/drive/MyDrive/hippo_encoder_runs')
TRAIN_JSONL = DATA_ROOT / 'train.jsonl'
RUN_DIR = RUN_ROOT / 'distill-bge-small-reset'

DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

print('TRAIN_JSONL =', TRAIN_JSONL)
print('RUN_DIR =', RUN_DIR)


In [ ]:
%cd /content/Hippo-encoder
!python scripts/prepare_text_dataset.py \
  --dataset ag_news \
  --split train \
  --text-column text \
  --limit 20000 \
  --prefix 'passage: ' \
  --output /content/drive/MyDrive/hippo_encoder_data/train.jsonl


In [ ]:
from pathlib import Path
import json

config = {
    'teacher_model_name': 'intfloat/e5-base-v2',
    'student_model_name': 'BAAI/bge-small-en-v1.5',
    'dataset_jsonl': '/content/drive/MyDrive/hippo_encoder_data/train.jsonl',
    'output_dir': '/content/drive/MyDrive/hippo_encoder_runs/distill-bge-small-reset',
    'max_text_length': 64,
    'batch_size': 32,
    'num_epochs': 3,
    'learning_rate': 1e-4,
    'weight_decay': 1e-2,
    'log_every': 20,
    'save_every': 100000,
    'num_workers': 2,
    'teacher_text_weight': 1.0,
    'hidden_state_weight': 0.2,
    'contrastive_weight': 0.0,
    'normalize_targets': True,
    'gradient_clip_norm': 1.0,
    'warmup_steps': 100,
    'seed': 42,
}

Path('/content/Hippo-encoder/configs/colab_run.json').write_text(
    json.dumps(config, indent=2),
    encoding='utf-8',
)
print('wrote /content/Hippo-encoder/configs/colab_run.json')


In [ ]:
%cd /content/Hippo-encoder
!python -m hippo_encoder.train --config configs/colab_run.json


In [ ]:
%cd /content/Hippo-encoder
!python scripts/eval_student_encoder.py \
  --student-checkpoint /content/drive/MyDrive/hippo_encoder_runs/distill-bge-small-reset/epoch-2


## Notes

- This reset notebook is intentionally limited to the original tiny-LLM encoder distillation path.
- The main success criteria are teacher-student cosine agreement and neighbor preservation.
- Region / delta experiments are still in the repo, but they are not part of this reset workflow.
- If you change `output_dir`, update the checkpoint path in the evaluation cell.
